# Build analysis tables

This notebook is part of the public reproducibility workflow for the study.

Public workflow stage: 06  
Purpose: Build core review-level and paper-level analysis tables while preserving the base-to-v2 lineage.

Source notebooks:
- `主表及分析表构建.ipynb`: cells 4–7 and 9–10 (base paper, review, author, aspect, and JSD tables)
- `分析.ipynb`: cells 1–3 and 5 (formal v2 enhancements)

Inputs: prior public work tables from stages 01, 03, 04, and 05, plus future `sample_papers.csv`.  
Outputs: base tables under `data/work/base_tables/` and the three formal v2 files under `data/work/`.

The formal direction is `paper_master` → `paper_author_master` → `paper_background_features` → v2. This notebook does not read author names, DOI, title, or raw affiliation into its public outputs.


In [ ]:
from pathlib import Path
import sys

_places = (Path.cwd().resolve(), Path.cwd().resolve().parent)
_root = next((p for p in _places if (p / "src" / "project_paths.py").is_file()), None)
if _root is None:
    raise RuntimeError("Start Jupyter from the public repository root or notebooks directory")
sys.path.insert(0, str(_root))
from src.project_paths import CONFIG_DIR, project_path

try:
    import yaml
except ImportError as exc:
    raise ImportError("PyYAML is required to read the example configuration") from exc
with (CONFIG_DIR / "config.example.yaml").open(encoding="utf-8") as _stream:
    config = yaml.safe_load(_stream)
sample_dir = project_path(config["paths"]["sample_data"])
work_dir = project_path(config["paths"]["work_data"])

from itertools import combinations
import numpy as np
import pandas as pd
from scipy.spatial.distance import jensenshannon

ASPECT_CLASSES = [
    "Methodology", "Results", "Experiments and Evaluation", "Data and Datasets",
    "Figures and Tables", "Writing, Language and Terminology", "Theory and Models",
    "Comparison and Context", "Paper Assessment", "Novelty and Impact",
    "Validity, Rigor and Quality", "Limitations",
]
assert len(ASPECT_CLASSES) == 12, "The frozen taxonomy must contain 12 classes"
COUNTRY_MIN_PAPERS = int(config["analysis"]["formal_country_min_papers"])
LARGE_TEAM_CUTOFF = int(config["analysis"]["large_team_cutoff"])
assert COUNTRY_MIN_PAPERS == 500, "Formal country threshold must remain 500"
assert LARGE_TEAM_CUTOFF == 50, "Formal large-team cutoff must remain 50"
base_dir = work_dir / "base_tables"
base_dir.mkdir(parents=True, exist_ok=True)

def require_parquet(relative):
    path = work_dir / relative
    if not path.is_file():
        raise FileNotFoundError(f"Required stage input is not available yet: {path}")
    return pd.read_parquet(path)

def join_unique(values):
    items = []
    for value in values:
        if isinstance(value, (list, tuple, set, np.ndarray)):
            items.extend(str(item).strip() for item in value if pd.notna(item) and str(item).strip())
        elif pd.notna(value) and str(value).strip():
            items.append(str(value).strip())
    return " | ".join(sorted(set(items)))

def first_nonempty(values):
    for value in values:
        if pd.notna(value) and str(value).strip():
            return str(value).strip()
    return ""

def safe_numeric(values):
    return pd.to_numeric(values, errors="coerce")

def entropy_from_counts(counts, eps=1e-12):
    array = np.asarray(counts, dtype=float)
    total = array.sum()
    if total <= 0:
        return np.nan
    probabilities = array / total
    probabilities = probabilities[probabilities > 0]
    return float(-(probabilities * np.log(probabilities + eps)).sum())


## Part A — Build base tables

The base tables follow the formal source ordering. The public copy retains the statistical columns needed for later stages and omits direct author identity, DOI, paper title, and raw review text from saved tables.


### A1. Paper master, then review master

Publication year prefers an available OpenAlex year and falls back to formal paper metadata. Subject codes and topics retain their paper-level aggregation rules.


In [ ]:
paper_meta = require_parquet("tables_and_vocab/paper_meta.parquet")
paper_subject_map = require_parquet("tables_and_vocab/paper_subject_map.parquet")
review_paper_map = require_parquet("tables_and_vocab/review_paper_map.parquet")
review_level = require_parquet("review_level.parquet")
sample_papers_path = sample_dir / "sample_papers.csv"
if not sample_papers_path.is_file():
    raise FileNotFoundError(f"Paper sample is not available yet: {sample_papers_path}")
works = pd.read_csv(sample_papers_path)
if "paper_code" not in works:
    raise KeyError("Paper sample must contain paper_code")
works = works.drop_duplicates("paper_code").copy()
subject_agg = paper_subject_map.groupby("paper_code", as_index=False).agg(
    subject_codes=("subject_code", join_unique),
    topic_list=("topic", join_unique),
    subject_year=("year", first_nonempty))
meta_cols = [column for column in ("paper_code", "pub_date", "year", "subject_codes", "topic_list")
             if column in paper_meta]
meta = paper_meta[meta_cols].copy()
for column in ("subject_codes", "topic_list"):
    if column in meta:
        meta[column] = meta[column].map(
            lambda value: " | ".join(map(str, value)) if isinstance(value, (list, tuple, np.ndarray)) else value)
work_cols = [column for column in (
    "paper_code", "openalex_found", "openalex_publication_year", "publication_year",
    "openalex_authorships_returned", "openalex_truncated_suspected") if column in works]
paper_master = works[work_cols].merge(meta, on="paper_code", how="outer", suffixes=("", "_meta"))
paper_master = paper_master.merge(subject_agg, on="paper_code", how="left", suffixes=("", "_from_subject_map"))
if "openalex_publication_year" in paper_master:
    year = safe_numeric(paper_master["openalex_publication_year"])
elif "publication_year" in paper_master:
    year = safe_numeric(paper_master["publication_year"])
else:
    year = pd.Series(np.nan, index=paper_master.index)
if "year" in paper_master:
    year = year.fillna(safe_numeric(paper_master["year"]))
if "subject_year" in paper_master:
    year = year.fillna(safe_numeric(paper_master["subject_year"]))
paper_master["publication_year"] = year.astype("Int64")
paper_master["subject_codes_final"] = paper_master["subject_codes"].fillna(
    paper_master["subject_codes_from_subject_map"]) if "subject_codes_from_subject_map" in paper_master else paper_master.get("subject_codes", "")
paper_master["topic_list_final"] = paper_master["topic_list"].fillna(
    paper_master["topic_list_from_subject_map"]) if "topic_list_from_subject_map" in paper_master else paper_master.get("topic_list", "")
keep = [column for column in (
    "paper_code", "openalex_found", "publication_year", "subject_codes_final",
    "topic_list_final", "openalex_authorships_returned", "openalex_truncated_suspected") if column in paper_master]
paper_master = paper_master[keep].drop_duplicates("paper_code").copy()
paper_master.to_parquet(base_dir / "paper_master_clean.parquet", index=False)

required_review = {"review_id", "paper_code", "reviewer_tag", "content_raw"}
if missing := required_review.difference(review_level.columns):
    raise KeyError(f"Review-level input is missing columns: {sorted(missing)}")
review_master = review_level[["review_id", "paper_code", "reviewer_tag", "content_raw"]].copy()
for column in ("review_id", "paper_code", "reviewer_tag"):
    review_master[column] = review_master[column].astype(str).str.strip()
review_master["review_char_len"] = review_master["content_raw"].fillna("").astype(str).str.len()
review_master["review_word_count"] = review_master["content_raw"].fillna("").astype(str).map(lambda value: len(value.split()))
review_master = review_master.drop(columns="content_raw")
review_master = review_master.merge(review_paper_map, on=["review_id", "paper_code"], how="left")
review_master = review_master.merge(paper_master, on="paper_code", how="left")
if "review_year" not in review_master:
    review_master["review_year"] = pd.NA
review_master["review_year"] = safe_numeric(review_master["review_year"]).astype("Int64")
review_master["review_year"] = review_master["review_year"].fillna(review_master["publication_year"])
review_master = review_master.drop_duplicates("review_id").copy()
review_master.to_parquet(base_dir / "review_master_clean.parquet", index=False)
print(f"Built {len(paper_master)} paper and {len(review_master)} review master rows")


### A2. Paper-author master and paper background

Notebook 05 has already applied the v1 gender patch and countryfill v3 source to the formal `paper_authors.parquet` branch. This section joins it to the newly built paper master before aggregating role and team features.


In [ ]:
paper_author = require_parquet("paper_authors.parquet")
country_source = require_parquet("countryfilled_with_source.parquet")
required_author = {"paper_code", "author_order", "is_first_author", "is_last_author",
                   "is_corresponding_author", "author_affil_country_primary",
                   "name_inferred_gender", "gender_status"}
if missing := required_author.difference(paper_author.columns):
    raise KeyError(f"Paper-author input is missing columns: {sorted(missing)}")
if "institution_country_source" not in paper_author:
    paper_author = paper_author.merge(
        country_source[["paper_code", "author_order", "institution_country_source"]],
        on=["paper_code", "author_order"], how="left")
paper_author_master = paper_author.merge(
    paper_master[["paper_code", "publication_year", "subject_codes_final", "topic_list_final"]],
    on="paper_code", how="left", validate="many_to_one")
paper_author_master = paper_author_master.drop(columns=["author_uid"], errors="ignore")
paper_author_master.to_parquet(base_dir / "paper_author_master.parquet", index=False)

def get_role_rows(group, role):
    if role == "first":
        return group[group["is_first_author"] == True] if group["is_first_author"].any() else group[group["author_order"] == group["author_order"].min()]
    if role == "last":
        return group[group["is_last_author"] == True] if group["is_last_author"].any() else group[group["author_order"] == group["author_order"].max()]
    if role == "corresponding":
        return group[group["is_corresponding_author"] == True] if group["is_corresponding_author"].any() else group.iloc[0:0]
    raise ValueError("Unknown author role")

def summarize_role(group, role):
    rows = get_role_rows(group, role)
    if rows.empty:
        return {f"{role}_author_country": "", f"{role}_author_gender": "",
                f"{role}_author_gender_status": "unknown", f"{role}_author_gender_probability": np.nan}
    return {f"{role}_author_country": join_unique(rows["author_affil_country_primary"]),
            f"{role}_author_gender": join_unique(rows["name_inferred_gender"]),
            f"{role}_author_gender_status": join_unique(rows["gender_status"]),
            f"{role}_author_gender_probability": float(safe_numeric(rows["gender_probability"]).max()) if "gender_probability" in rows else np.nan}

def summarize_paper_authors(group):
    out = {"paper_code": group["paper_code"].iloc[0], "n_authors": len(group)}
    countries = [value for value in group["author_affil_country_primary"].fillna("").astype(str) if value.strip()]
    counts = pd.Series(countries).value_counts() if countries else pd.Series(dtype=int)
    out["n_author_countries"] = len(set(countries))
    out["dominant_country"] = counts.index[0] if len(counts) else ""
    out["is_international_collaboration"] = int(out["n_author_countries"] >= 2)
    for role in ("first", "last", "corresponding"):
        out.update(summarize_role(group, role))
    gender = group["name_inferred_gender"].fillna("").astype(str).str.lower()
    status = group["gender_status"].fillna("unknown").astype(str)
    known = gender.isin(["male", "female"])
    confident = known & status.eq("confident")
    out["n_gender_known"] = int(known.sum())
    out["n_gender_confident"] = int(confident.sum())
    out["n_female_inferred"] = int(gender.eq("female").sum())
    out["n_male_inferred"] = int(gender.eq("male").sum())
    out["n_female_confident"] = int((gender.eq("female") & confident).sum())
    out["n_male_confident"] = int((gender.eq("male") & confident).sum())
    out["female_author_ratio"] = out["n_female_inferred"] / out["n_gender_known"] if out["n_gender_known"] else np.nan
    out["female_author_ratio_confident"] = out["n_female_confident"] / out["n_gender_confident"] if out["n_gender_confident"] else np.nan
    out["gender_unknown_ratio"] = 1 - out["n_gender_known"] / out["n_authors"] if out["n_authors"] else np.nan
    out["has_first_author_country"] = int(bool(out["first_author_country"].strip()))
    out["has_corresponding_author_country"] = int(bool(out["corresponding_author_country"].strip()))
    out["has_confident_gender"] = int(out["n_gender_confident"] > 0)
    return pd.Series(out)

paper_background = (paper_author_master.groupby("paper_code", as_index=False)
                    .apply(summarize_paper_authors).reset_index(drop=True))
paper_background = paper_background.merge(paper_master, on="paper_code", how="left", validate="one_to_one")
paper_background.to_parquet(base_dir / "paper_background_features.parquet", index=False)
print(f"Built {len(paper_author_master)} paper-author and {len(paper_background)} paper-background rows")


### A3. Review class counts and paper disagreement

Only mapped terms contribute to class counts. Review class ratios use `aspect_total_count` as the denominator. A paper's JSD is computed only from reviews with at least one mapped aspect; SciPy's Jensen–Shannon distance and the source aggregation are retained.


In [ ]:
review_term_norm = require_parquet("review_term_counts_normalized.parquet")
valid = review_term_norm[review_term_norm["major_class_final"].notna() &
                         review_term_norm["major_class_final"].astype(str).str.len().gt(0)].copy()
valid["major_class_final"] = valid["major_class_final"].astype(str).str.strip()
if unexpected := set(valid["major_class_final"]) - set(ASPECT_CLASSES):
    raise ValueError(f"Unexpected aspect class in normalized terms: {sorted(unexpected)}")
review_class_long = valid.groupby(["review_id", "paper_code", "major_class_final"], as_index=False).agg(
    class_count=("count_in_review", "sum"), n_terms=("term_norm", "nunique"))
review_class_wide = review_class_long.pivot_table(
    index=["review_id", "paper_code"], columns="major_class_final", values="class_count",
    aggfunc="sum", fill_value=0).reset_index()
for aspect_class in ASPECT_CLASSES:
    if aspect_class not in review_class_wide:
        review_class_wide[aspect_class] = 0
review_class_wide = review_class_wide[["review_id", "paper_code"] + ASPECT_CLASSES]
review_class_wide["aspect_total_count"] = review_class_wide[ASPECT_CLASSES].sum(axis=1)
review_meta_cols = [column for column in (
    "review_id", "paper_code", "reviewer_tag", "review_year", "publication_year",
    "subject_codes_final", "topic_list_final", "review_char_len", "review_word_count")
    if column in review_master]
review_counts = review_class_wide.merge(
    review_master[review_meta_cols].drop_duplicates("review_id"),
    on=["review_id", "paper_code"], how="left")
background_cols = [column for column in paper_background if column != "publication_year"]
review_counts = review_counts.merge(paper_background[background_cols], on="paper_code", how="left", suffixes=("", "_paperbg"))
review_counts.to_parquet(base_dir / "review_aspect_class_counts.parquet", index=False)
review_ratios = review_counts.copy()
for aspect_class in ASPECT_CLASSES:
    review_ratios[aspect_class] = np.where(
        review_ratios["aspect_total_count"] > 0,
        review_ratios[aspect_class] / review_ratios["aspect_total_count"], np.nan)
review_ratios.to_parquet(base_dir / "review_aspect_class_ratios.parquet", index=False)

def compute_jsd_metrics(group, aspect_cols):
    selected = group[group[aspect_cols].sum(axis=1) > 0].copy()
    result = {"n_reviews": len(group), "n_reviews_valid_for_jsd": len(selected),
              "mean_pairwise_jsd": np.nan, "max_pairwise_jsd": np.nan,
              "centroid_jsd": np.nan, "review_aspect_entropy_mean": np.nan,
              "review_aspect_entropy_sd": np.nan}
    if selected.empty:
        return result
    counts = selected[aspect_cols].astype(float).values
    probabilities = counts / counts.sum(axis=1, keepdims=True)
    entropies = [entropy_from_counts(row) for row in counts]
    result["review_aspect_entropy_mean"] = float(np.nanmean(entropies))
    result["review_aspect_entropy_sd"] = float(np.nanstd(entropies))
    if len(selected) < 2:
        return result
    pairwise = [float(jensenshannon(probabilities[i], probabilities[j]))
                for i, j in combinations(range(len(selected)), 2)]
    centroid = probabilities.mean(axis=0)
    centroid = centroid / centroid.sum()
    result["mean_pairwise_jsd"] = float(np.mean(pairwise))
    result["max_pairwise_jsd"] = float(np.max(pairwise))
    result["centroid_jsd"] = float(np.mean([
        jensenshannon(probability, centroid) for probability in probabilities]))
    return result

paper_rows = []
for paper_code, group in review_counts.groupby("paper_code"):
    class_counts = group[ASPECT_CLASSES].sum()
    total = class_counts.sum()
    row = {"paper_code": paper_code, "aspect_total_count": total}
    for aspect_class in ASPECT_CLASSES:
        row[aspect_class] = class_counts[aspect_class]
        row[f"{aspect_class}__ratio"] = class_counts[aspect_class] / total if total > 0 else np.nan
    row.update(compute_jsd_metrics(group, ASPECT_CLASSES))
    paper_rows.append(row)
paper_jsd = pd.DataFrame(paper_rows).merge(paper_background, on="paper_code", how="left", suffixes=("", "_bg"))
paper_jsd = paper_jsd.merge(paper_master, on="paper_code", how="left", suffixes=("", "_paper"))
paper_jsd.to_parquet(base_dir / "paper_aspect_disagreement.parquet", index=False)
print(f"Built {len(review_counts)} review class rows and {len(paper_jsd)} paper disagreement rows")


## Part B — Apply the formal v2 enrichments

The base paper background is expanded with fixed large-team/capping rules, country missingness, role confidence, and gender combinations. The country threshold of 500 is reserved for the formal country grouping in the downstream strict analysis; it is not lowered to make a small demo appear complete.


In [ ]:
def clean_gender_value(value):
    if pd.isna(value):
        return ""
    value = str(value).strip().lower()
    if value in {"male", "m"}:
        return "male"
    if value in {"female", "f", "woman", "women"}:
        return "female"
    return ""

def gender_to_letter(value):
    return {"male": "M", "female": "W"}.get(clean_gender_value(value), "")

def make_country_group(series, top_n=10, min_count=None, other_label="Other", unknown_label="Unknown"):
    values = series.fillna("").astype(str).str.strip().replace({"": unknown_label, "nan": unknown_label, "None": unknown_label})
    counts = values.value_counts()
    keep = counts[counts >= min_count].index.tolist() if min_count is not None else counts.head(top_n).index.tolist()
    keep = [value for value in keep if value != unknown_label]
    return values.map(lambda value: value if value in keep or value == unknown_label else other_label), keep

def first_valid_country_string(value):
    if pd.isna(value):
        return ""
    parts = [part.strip() for part in str(value).split("|") if part.strip()]
    return parts[0] if parts else ""

paper_bg2 = paper_background.copy()
p95 = paper_bg2["n_authors"].quantile(0.95)
p99 = paper_bg2["n_authors"].quantile(0.99)
paper_bg2["large_team_ge_30"] = paper_bg2["n_authors"].ge(30).astype(int)
paper_bg2["large_team_ge_50"] = paper_bg2["n_authors"].ge(LARGE_TEAM_CUTOFF).astype(int)
paper_bg2["large_team_top5pct"] = paper_bg2["n_authors"].ge(p95).astype(int)
paper_bg2["large_team_top1pct"] = paper_bg2["n_authors"].ge(p99).astype(int)
paper_bg2["author_count_capped_100"] = paper_bg2["n_authors"].ge(100).astype(int)
paper_bg2["openalex_truncated_suspected"] = paper_bg2.get("openalex_truncated_suspected", pd.Series(False, index=paper_bg2.index)).fillna(False).astype(bool).astype(int)
returned = safe_numeric(paper_bg2.get("openalex_authorships_returned", pd.Series(np.nan, index=paper_bg2.index)))
paper_bg2["openalex_authorships_returned_100"] = returned.ge(100).astype(int)
paper_bg2["openalex_truncated_or_capped"] = (
    paper_bg2["author_count_capped_100"].eq(1) |
    paper_bg2["openalex_truncated_suspected"].eq(1) |
    paper_bg2["openalex_authorships_returned_100"].eq(1)).astype(int)
paper_bg2["exclude_large_or_capped_team"] = paper_bg2["openalex_truncated_or_capped"]
paper_bg2["exclude_large_team_ge_50"] = paper_bg2["large_team_ge_50"]

pa_country = paper_author_master.copy()
pa_country["has_author_country"] = pa_country["author_affil_country_primary"].fillna("").astype(str).str.strip().str.len().gt(0).astype(int)
country_missing = pa_country.groupby("paper_code", as_index=False).agg(
    n_author_rows=("author_order", "count"), n_author_country_known=("has_author_country", "sum"))
country_missing["country_missing_ratio"] = 1 - country_missing["n_author_country_known"] / country_missing["n_author_rows"]
paper_bg2 = paper_bg2.merge(country_missing[["paper_code", "country_missing_ratio"]], on="paper_code", how="left")
paper_bg2["country_missing_ratio"] = paper_bg2["country_missing_ratio"].fillna(1.0)
paper_bg2["corresponding_author_country_primary"] = paper_bg2["corresponding_author_country"].map(first_valid_country_string)
for source, target in (("first_author_country", "first_author_country_group"),
                       ("corresponding_author_country_primary", "corresponding_author_country_group"),
                       ("dominant_country", "dominant_country_group")):
    paper_bg2[target], _ = make_country_group(paper_bg2[source], top_n=10)
for role in ("first", "last", "corresponding"):
    gender_col, status_col = f"{role}_author_gender", f"{role}_author_gender_status"
    paper_bg2[f"{role}_author_gender_confident"] = np.where(
        paper_bg2[status_col].fillna("").astype(str).str.contains("confident", case=False),
        paper_bg2[gender_col].map(clean_gender_value), "")

def simplify_corresponding_gender(value):
    if pd.isna(value) or not str(value).strip():
        return ""
    parts = sorted(set(filter(None, (clean_gender_value(part) for part in str(value).split("|")))))
    return parts[0] if len(parts) == 1 else "mixed_corresponding" if len(parts) > 1 else ""

paper_bg2["corresponding_author_gender_confident_simple"] = paper_bg2[
    "corresponding_author_gender_confident"].map(simplify_corresponding_gender)
paper_bg2["first_last_gender_pair"] = paper_bg2.apply(lambda row:
    gender_to_letter(row["first_author_gender_confident"]) + gender_to_letter(row["last_author_gender_confident"])
    if gender_to_letter(row["first_author_gender_confident"]) and gender_to_letter(row["last_author_gender_confident"])
    else "Unknown", axis=1)
paper_bg2["first_last_gender_pair_label"] = paper_bg2["first_last_gender_pair"].map({
    "MM": "first_male_last_male", "MW": "first_male_last_female",
    "WM": "first_female_last_male", "WW": "first_female_last_female"}).fillna("Unknown")

def full_team_gender(row):
    n_authors, n_conf = row["n_authors"], row["n_gender_confident"]
    if pd.isna(n_authors) or pd.isna(n_conf) or n_authors <= 0 or n_conf < n_authors:
        return "incomplete_gender_info"
    if row["n_female_confident"] == n_authors:
        return "all_female"
    if row["n_male_confident"] == n_authors:
        return "all_male"
    return "mixed" if row["n_female_confident"] > 0 and row["n_male_confident"] > 0 else "incomplete_gender_info"

def observed_team_gender(row):
    if pd.isna(row["n_gender_confident"]) or row["n_gender_confident"] <= 0:
        return "unknown"
    if row["n_female_confident"] > 0 and row["n_male_confident"] > 0:
        return "mixed_observed"
    if row["n_female_confident"] > 0:
        return "female_only_observed"
    return "male_only_observed" if row["n_male_confident"] > 0 else "unknown"

paper_bg2["team_gender_full_confident"] = paper_bg2.apply(full_team_gender, axis=1)
paper_bg2["team_gender_observed_confident"] = paper_bg2.apply(observed_team_gender, axis=1)
paper_bg2.to_parquet(work_dir / "paper_background_features_v2.parquet", index=False)


### B2. Carry the same v2 background into review counts and paper disagreement

Before the left join, prior columns with the same v2 names are removed. This reproduces the source's duplicate-column handling without recalculating any aspect counts or JSD values.


In [ ]:
bg_cols_to_add = [column for column in (
    "paper_code", "n_authors", "n_author_countries", "is_international_collaboration",
    "country_missing_ratio", "large_team_ge_30", "large_team_ge_50",
    "large_team_top5pct", "large_team_top1pct", "author_count_capped_100",
    "openalex_truncated_or_capped", "exclude_large_or_capped_team",
    "exclude_large_team_ge_50", "first_author_country", "corresponding_author_country",
    "corresponding_author_country_primary", "dominant_country",
    "first_author_country_group", "corresponding_author_country_group",
    "dominant_country_group", "first_author_gender_confident",
    "last_author_gender_confident", "corresponding_author_gender_confident_simple",
    "first_last_gender_pair", "first_last_gender_pair_label",
    "female_author_ratio_confident", "gender_unknown_ratio",
    "team_gender_full_confident", "team_gender_observed_confident") if column in paper_bg2]
review_counts2 = review_counts.drop(columns=[column for column in bg_cols_to_add
                                              if column != "paper_code" and column in review_counts])
review_counts2 = review_counts2.merge(paper_bg2[bg_cols_to_add], on="paper_code", how="left", validate="many_to_one")
review_counts2.to_parquet(work_dir / "review_aspect_class_counts_v2.parquet", index=False)
paper_jsd2 = paper_jsd.drop(columns=[column for column in bg_cols_to_add
                                      if column != "paper_code" and column in paper_jsd])
paper_jsd2 = paper_jsd2.merge(paper_bg2[bg_cols_to_add], on="paper_code", how="left", validate="one_to_one")
paper_jsd2.to_parquet(work_dir / "paper_aspect_disagreement_v2.parquet", index=False)
print("Saved the three formal v2 analysis tables")
